# exp001 - LLaMA 3.2 Stage 1 Baseline v2.1

**实验目标**：测量 LLaMA 3.2 系列模型（1B / 3B / 11B-Vision）在 Google Colab GPU 上的推理性能基线。

**测量指标**：TTFT（首 token 时延）/ TPOT（每 token 时延）/ KV Cache 大小 / 峰値显存

**版本**：v2.1 -- 支持 text_only 和 single_image 两种输入模式，forward hook 分项计时视觉编码。

## Section 0: Environment & Dependencies

In [ ]:
# Install required dependencies
# transformers: model loading and inference
# accelerate: device_map='auto' for GPU placement
# huggingface_hub: model download and auth
# ipywidgets: interactive configuration UI
# psutil: system memory monitoring
# pillow: image processing
# datasets: quality evaluation datasets
!pip install -q transformers accelerate huggingface_hub \
             ipywidgets psutil pillow datasets

In [ ]:
# ================================================================
# Path constants -- all paths defined here, not hardcoded elsewhere
# ================================================================
import sys
from pathlib import Path

# Google Drive project root (after mounting)
DRIVE_ROOT      = "/content/drive/MyDrive/EdgeLLM-Systems"

# Model weights cache (persisted on Drive to avoid re-downloading)
MODEL_CACHE_DIR = f"{DRIVE_ROOT}/models/llama32_models"

# Dataset directory (profiling_suite and quality_suite)
DATASET_DIR     = f"{DRIVE_ROOT}/dataset"

# Experiment results output directory
RESULTS_DIR     = f"{DRIVE_ROOT}/results/exp001"

# Project module path (edge_llm_systems added to sys.path after Drive mount)
PROJECT_SRC     = f"{DRIVE_ROOT}"
if PROJECT_SRC not in sys.path:
    sys.path.insert(0, PROJECT_SRC)

print(f"DRIVE_ROOT:      {DRIVE_ROOT}")
print(f"MODEL_CACHE_DIR: {MODEL_CACHE_DIR}")
print(f"DATASET_DIR:     {DATASET_DIR}")
print(f"RESULTS_DIR:     {RESULTS_DIR}")

## Section 1: Mount Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

# Mount Google Drive -- triggers auth dialog
drive.mount("/content/drive")

# Verify project root exists, create if needed
drive_root_path = Path(DRIVE_ROOT)
if not drive_root_path.exists():
    drive_root_path.mkdir(parents=True, exist_ok=True)
    print(f"[Drive] Created: {DRIVE_ROOT}")
else:
    print(f"[Drive] Exists: {DRIVE_ROOT}")

# 只确保 results/exp001/ 基础目录存在
# raw_runs/ 和 group_summary/ 在 Section 6 按模型名动态创建，不在这里预建
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

print("[Drive] Mount complete, directory structure ready")

## Section 2: HuggingFace Authentication

In [ ]:
from google.colab import userdata
from huggingface_hub import login

# Read HF Token from Colab Secrets (set via key icon in left sidebar)
# LLaMA 3.2 is gated -- accept terms on HuggingFace website first
HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("[HuggingFace] Authentication complete")

## Section 3: Experiment Configuration (Interactive UI)

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Model selector
w_model = widgets.Dropdown(
    options=["LLaMA-3.2-1B", "LLaMA-3.2-3B", "LLaMA-3.2-11B-Vision"],
    value="LLaMA-3.2-1B",
    description="Model:",
    style={"description_width": "initial"},
)
# Input mode: text_only or single_image
w_input_mode = widgets.Dropdown(
    options=["text_only", "single_image"],
    value="text_only",
    description="Input Mode:",
    style={"description_width": "initial"},
)
# Test type checkboxes
w_run_perf = widgets.Checkbox(value=True, description="Performance Test (Profiling)")
w_run_qual = widgets.Checkbox(value=False, description="Quality Evaluation")
# prompt_lengths (text_only)
w_prompt_lens = widgets.Text(
    value="64, 128, 256, 512, 1024, 2048",
    description="Prompt Lengths:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
# gen_lengths (common)
w_gen_lens = widgets.Text(
    value="32, 64, 128",
    description="Gen Lengths:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
# image_resolutions (single_image)
w_img_resolutions = widgets.Text(
    value="224, 336, 448",
    description="Image Resolutions:",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="400px"),
)
# repeat count
w_repeat = widgets.Dropdown(
    options=[1, 3, 5], value=3,
    description="Repeat:",
    style={"description_width": "initial"},
)
w_confirm = widgets.Button(
    description="Confirm Config",
    button_style="success",
    layout=widgets.Layout(width="150px"),
)
w_output = widgets.Output()

# Mutually exclusive display: text params vs image params
def _on_mode_change(change):
    if change["new"] == "text_only":
        w_prompt_lens.layout.display = ""
        w_img_resolutions.layout.display = "none"
    else:
        w_prompt_lens.layout.display = "none"
        w_img_resolutions.layout.display = ""

w_input_mode.observe(_on_mode_change, names="value")
w_img_resolutions.layout.display = "none"  # initial state: hidden

# Global config dict used by subsequent cells
CONFIG = {}

def _on_confirm(b):
    global CONFIG
    with w_output:
        w_output.clear_output()
        def parse_ints(s):
            return [int(x.strip()) for x in s.split(",") if x.strip()]
        CONFIG = {
            "model_key":         w_model.value,
            "input_mode":        w_input_mode.value,
            "run_performance":   w_run_perf.value,
            "run_quality":       w_run_qual.value,
            "prompt_lengths":    parse_ints(w_prompt_lens.value),
            "gen_lengths":       parse_ints(w_gen_lens.value),
            "image_resolutions": parse_ints(w_img_resolutions.value),
            "repeat":            w_repeat.value,
        }
        yes = "\u662f"  # Chinese: Yes
        no  = "\u5426"  # Chinese: No
        print("\u2705 \u914d\u7f6e\u5df2\u786e\u8ba4\uff1a")  # Config confirmed
        print(f"   \u6a21\u578b\uff1a{CONFIG['model_key']}")
        print(f"   \u8f93\u5165\u6a21\u5f0f\uff1a{CONFIG['input_mode']}")
        print(f"   \u6027\u80fd\u6d4b\u8bd5\uff1a{yes if CONFIG['run_performance'] else no}")
        print(f"   \u8d28\u91cf\u8bc4\u4f30\uff1a{yes if CONFIG['run_quality'] else no}")
        if CONFIG["input_mode"] == "text_only":
            print(f"   Prompt Lengths: {CONFIG['prompt_lengths']}")
        else:
            print(f"   Image Resolutions: {CONFIG['image_resolutions']}")
        print(f"   Gen Lengths: {CONFIG['gen_lengths']}")
        print(f"   \u91cd\u590d\u6b21\u6570\uff1a{CONFIG['repeat']}")
        if CONFIG['run_performance'] and not CONFIG['run_quality']:
            print("   \u26a0\ufe0f \u5f53\u524d\u4ec5\u8fd0\u884c\u6027\u80fd\u6d4b\u8bd5\uff0c\u8d28\u91cf\u8bc4\u4f30\u672a\u542f\u7528")
        elif CONFIG['run_quality'] and not CONFIG['run_performance']:
            print("   \u26a0\ufe0f \u5f53\u524d\u4ec5\u8fd0\u884c\u8d28\u91cf\u8bc4\u4f30\uff0c\u6027\u80fd\u6d4b\u8bd5\u672a\u542f\u7528")

w_confirm.on_click(_on_confirm)

display(widgets.VBox([
    widgets.HBox([w_model, w_input_mode]),
    widgets.HBox([w_run_perf, w_run_qual]),
    w_prompt_lens,
    w_img_resolutions,
    w_gen_lens,
    w_repeat,
    w_confirm,
    w_output,
]))

## Section 4: Model Loading

In [ ]:
import torch
from edge_llm_systems.models import (
    check_model_integrity, download_model_if_needed,
    load_text_model, load_vision_model,
    get_model_load_mem_mb, MODEL_REGISTRY,
)
from edge_llm_systems.utils import log

assert CONFIG, "Please run Section 3 and click Confirm first"

model_key  = CONFIG["model_key"]
input_mode = CONFIG["input_mode"]
device     = "cuda" if torch.cuda.is_available() else "cpu"

log(f"[Model Load] Target: {model_key}")
log(f"[Model Load] Mode: {input_mode}")
log(f"[Model Load] Device: {device}")

# Integrity check + conditional download (cached on Drive)
local_model_path = download_model_if_needed(
    model_key=model_key,
    cache_dir=MODEL_CACHE_DIR,
    hf_token=HF_TOKEN,
)

# Load model based on input mode
# NOTE: LLaMA-3.2-11B-Vision FP16 OOM records oom_stage='model_load', NO fallback
if input_mode == "text_only":
    model, tokenizer = load_text_model(local_model_path, HF_TOKEN)
    processor = None
else:  # single_image
    model, processor = load_vision_model(local_model_path, HF_TOKEN)
    tokenizer = processor.tokenizer

# Record baseline GPU memory (model weights only, before any inference)
model_load_mem_mb = get_model_load_mem_mb(device)

cfg = model.config
num_layers   = getattr(cfg, "num_hidden_layers", "N/A")
num_kv_heads = getattr(cfg, "num_key_value_heads",
                        getattr(cfg, "num_attention_heads", "N/A"))

log(f"  Path:     {local_model_path}")
log(f"  Device:   {model.device}")
log(f"  Layers:   {num_layers}")
log(f"  KV Heads: {num_kv_heads}")
log(f"  Base Mem: {model_load_mem_mb:.1f} MB")

## Section 5: Warm-up

In [ ]:
from edge_llm_systems.runners import run_warmup_text, run_warmup_vision
from edge_llm_systems.utils import log

log("[Warmup] Starting (results NOT written to CSV)...")

if input_mode == "text_only":
    # Short sequences to warm up CUDA kernels and cuDNN autotuning
    run_warmup_text(model=model, tokenizer=tokenizer, device=device)
else:
    # Vision warmup: text first, then image
    from PIL import Image as PILImage
    # Solid-color image -- ensures image preprocessing path is also warmed up
    warmup_image = PILImage.new("RGB", (224, 224), color=(128, 128, 128))
    run_warmup_vision(model=model, processor=processor, device=device,
                      warmup_image=warmup_image)

log("[Warmup] Complete")

## Section 6: Performance Benchmarking

In [ ]:
from pathlib import Path
from edge_llm_systems.runners import run_benchmark_text, run_benchmark_vision
from edge_llm_systems.utils import (
    log, generate_run_id, build_timestamp_filename, model_hash,
    collect_environment_info, collect_model_info, collect_run_config, save_json,
)
from edge_llm_systems.models import MODEL_REGISTRY

RUN_PERFORMANCE = CONFIG.get("run_performance", True)

if not RUN_PERFORMANCE:
    print("[Profiling] Not enabled, skipping")
else:
    run_id   = generate_run_id()
    model_id = MODEL_REGISTRY[model_key]

    # model_slug 取 HuggingFace repo id 的最后一段，用作结果子目录名
    # 例：meta-llama/Llama-3.2-1B-Instruct → Llama-3.2-1B-Instruct
    model_slug = model_id.split("/")[-1]

    # ── 结果目录（按模型名隔离，避免不同模型结果混放）────────────────
    # My Drive/EdgeLLM-Systems/results/exp001/{model_slug}/
    # ├── raw_runs/
    # ├── group_summary/
    # └── exp_info/
    #     ├── environment/
    #     ├── model/
    #     └── run_config/
    model_result_dir = Path(f"{RESULTS_DIR}/{model_slug}")
    raw_runs_dir     = model_result_dir / "raw_runs"
    summary_dir      = model_result_dir / "group_summary"
    env_info_dir     = model_result_dir / "exp_info" / "environment"
    model_info_dir   = model_result_dir / "exp_info" / "model"
    run_cfg_dir      = model_result_dir / "exp_info" / "run_config"

    for d in [raw_runs_dir, summary_dir, env_info_dir, model_info_dir, run_cfg_dir]:
        d.mkdir(parents=True, exist_ok=True)

    # 带时间戳的文件名，防止覆盖历史数据
    raw_csv_path     = str(raw_runs_dir    / build_timestamp_filename("raw_runs",     "csv"))
    summary_csv_path = str(summary_dir     / build_timestamp_filename("group_summary","csv"))

    # ── 保存 exp_info JSON 三件套 ─────────────────────────────────────
    # 1. environment.json：GPU 型号、CUDA 版本、PyTorch/transformers 版本等
    env_info = collect_environment_info(
        run_id=run_id, model_id=model_id, model_key=model_key,
        raw_csv_path=raw_csv_path, summary_csv_path=summary_csv_path,
    )
    env_path = env_info_dir / f"{run_id}_environment.json"
    save_json(env_path, env_info)
    log(f"[Exp Info] environment → {env_path.name}")

    # 2. model_info.json：层数、KV 头数、参数量、每千 token KV Cache 大小等
    #    同一模型只存一份（文件已存在则跳过，避免重复写入）
    model_info_path = model_info_dir / f"{model_slug}_model_info.json"
    if not model_info_path.exists():
        model_info = collect_model_info(
            model=model, model_id=model_id, model_key=model_key,
            local_model_path=local_model_path,
        )
        save_json(model_info_path, model_info)
        log(f"[Exp Info] model_info → {model_info_path.name}")
    else:
        log(f"[Exp Info] model_info already exists, skipped")

    # 3. run_config.json：本次实验的完整参数配置，含参数矩阵列表
    run_cfg = collect_run_config(
        run_id=run_id, model_id=model_id, model_key=model_key,
        config=CONFIG, raw_csv_path=raw_csv_path, summary_csv_path=summary_csv_path,
    )
    run_cfg_path = run_cfg_dir / f"{run_id}_run_config.json"
    save_json(run_cfg_path, run_cfg)
    log(f"[Exp Info] run_config → {run_cfg_path.name}")

    # ── 每行 CSV 附加的元数据 ─────────────────────────────────────────
    run_meta = {
        "run_id":     run_id,
        "model_id":   model_id,
        "model_hash": model_hash(model_id),
        "input_mode": input_mode,
        "test_type":  "profiling",
    }

    log(f"[Profiling] run_id:      {run_id}")
    log(f"[Profiling] model_slug:  {model_slug}")
    log(f"[Profiling] raw CSV:     {raw_csv_path}")
    log(f"[Profiling] summary CSV: {summary_csv_path}")

    if input_mode == "text_only":
        run_benchmark_text(
            model=model, tokenizer=tokenizer, device=device,
            prompt_lengths=CONFIG["prompt_lengths"],
            gen_lengths=CONFIG["gen_lengths"],
            repeat=CONFIG["repeat"],
            model_load_mem_mb=model_load_mem_mb,
            raw_csv_path=raw_csv_path,
            summary_csv_path=summary_csv_path,
            run_meta=run_meta,
        )
    else:  # single_image
        from PIL import Image as PILImage

        images_by_resolution = {}
        vqa_path = Path(f"{DATASET_DIR}/quality_suite/multimodal/vqav2_mini")

        for resolution in CONFIG["image_resolutions"]:
            image_files = (list(vqa_path.glob("*.jpg")) +
                           list(vqa_path.glob("*.png")))
            if image_files:
                img = PILImage.open(image_files[0]).convert("RGB")
            else:
                log(f"[Warning] No VQAv2 images found, using placeholder")
                img = PILImage.new("RGB", (resolution, resolution),
                                   color=(100, 150, 200))
            # 固定 LANCZOS resize，不使用随机变换，保证可复现
            images_by_resolution[resolution] = img.resize(
                (resolution, resolution), PILImage.LANCZOS)

        run_benchmark_vision(
            model=model, processor=processor, device=device,
            images_by_resolution=images_by_resolution,
            gen_lengths=CONFIG["gen_lengths"],
            repeat=CONFIG["repeat"],
            model_load_mem_mb=model_load_mem_mb,
            raw_csv_path=raw_csv_path,
            summary_csv_path=summary_csv_path,
            run_meta=run_meta,
        )

## Section 7: Quality Evaluation

Evaluates model accuracy on standard benchmarks.

**Text-only benchmarks (5 total):**
- **MMLU-Pro mini** (70 samples): College-level MCQ, A/B/C/D choice
- **GSM8K mini** (50 samples): Math word problems, extract final number
- **HellaSwag mini** (50 samples): Commonsense reasoning, sentence completion
- **WinoGrande mini** (50 samples): Pronoun resolution, binary choice
- **TruthfulQA-MC** (50 samples): Factual MCQ

**Single-image benchmarks (5 total):**
- **VQAv2 mini** (50 samples): General VQA, soft exact match
- **MMBench mini** (50 samples): Multi-dimensional vision MCQ
- **MathVista mini** (30 samples): Visual math reasoning, MCQ + numeric
- **TextVQA mini** (30 samples): OCR-based VQA, soft match
- **DocVQA mini** (30 samples): Document VQA, ANLS score

In [ ]:
import json
import re
from pathlib import Path
from datasets import load_dataset


def _load_text_dataset(dataset_dir, benchmark_name, hf_dataset_id,
                        hf_split, max_samples, sample_key="id"):
    """Check local cache; download from HuggingFace if missing/stale."""
    cache_dir = Path(dataset_dir) / "quality_suite" / "text_only" / benchmark_name
    manifest_path = cache_dir / "manifest.json"
    if manifest_path.exists():
        with open(manifest_path, encoding="utf-8") as f:
            m = json.load(f)
        if m.get("sample_count") == max_samples:
            data_path = cache_dir / "data.json"
            if data_path.exists():
                with open(data_path, encoding="utf-8") as f:
                    return json.load(f)
    print(f"[Dataset] Downloading {benchmark_name} from {hf_dataset_id}...")
    ds = load_dataset(hf_dataset_id, split=hf_split, trust_remote_code=True)
    samples = [dict(ds[i]) for i in range(min(max_samples, len(ds)))]
    cache_dir.mkdir(parents=True, exist_ok=True)
    with open(cache_dir / "data.json", "w", encoding="utf-8") as f:
        json.dump(samples, f, ensure_ascii=False, indent=2)
    manifest = {
        "benchmark": benchmark_name, "source": hf_dataset_id,
        "split": hf_split, "sample_count": len(samples),
        "sample_ids": [str(s.get(sample_key, i)) for i, s in enumerate(samples)],
    }
    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(f"[Dataset] {benchmark_name}: {len(samples)} samples cached")
    return samples


def _gen_text_answer(model, tokenizer, device, prompt, max_new_tokens):
    """Greedy decoding: temperature=0, do_sample=False."""
    import torch
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=False, pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out[0, inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


def _extract_mc(text):
    """Extract A/B/C/D letter from generated text."""
    text = text.strip().upper()
    m = re.search(r'(?:ANSWER|THE ANSWER IS)[\s:]*([A-D])', text)
    if m: return m.group(1)
    m = re.search(r'\b([A-D])\b', text)
    return m.group(1) if m else ""


def _extract_num(text):
    """Extract the last number from text (for GSM8K)."""
    nums = re.findall(r'-?\d+(?:\.\d+)?', text)
    return nums[-1] if nums else ""


print("[Quality] Helper functions ready")

In [ ]:
import torch
from edge_llm_systems.utils import log

RUN_QUALITY = CONFIG.get("run_quality", False)
input_mode  = CONFIG["input_mode"]

if not RUN_QUALITY:
    print("[Quality] Not enabled, skipping")
elif input_mode != "text_only":
    print("[Quality] Vision mode -- text benchmarks skipped")
else:

    def run_mmlu_pro_mini(model, tokenizer, device, dataset_dir, max_samples=70):
        """MMLU-Pro mini: college-level MCQ, 70 samples."""
        samples = _load_text_dataset(
            dataset_dir, "mmlu_pro_mini", "TIGER-Lab/MMLU-Pro", "test", max_samples)
        correct = answered = 0
        for s in samples:
            opts = "\n".join(f"{chr(65+i)}. {o}"
                             for i, o in enumerate(s.get("options", [])))
            prompt = f"Question: {s.get('question', '')}\n{opts}\nAnswer:"
            pred = _extract_mc(_gen_text_answer(model, tokenizer, device, prompt, 10))
            if pred:
                answered += 1
                if pred == s.get("answer", "").upper(): correct += 1
        score = correct / len(samples) * 100 if samples else 0
        ans_rate = answered / len(samples) * 100 if samples else 0
        print(f"MMLU-Pro mini: {correct}/{len(samples)} = {score:.1f}%"
              f" | answered={ans_rate:.0f}%")
        return {"benchmark": "mmlu_pro_mini", "score": score, "answered": ans_rate}

    def run_gsm8k_mini(model, tokenizer, device, dataset_dir, max_samples=50):
        """GSM8K mini: math word problems, 50 samples."""
        samples = _load_text_dataset(
            dataset_dir, "gsm8k_mini", "openai/gsm8k", "test", max_samples)
        correct = answered = 0
        for s in samples:
            gold = _extract_num(s.get("answer", ""))
            prompt = f"Solve step by step.\nQ: {s.get('question', '')}\nA:"
            pred = _extract_num(_gen_text_answer(model, tokenizer, device, prompt, 256))
            if pred:
                answered += 1
                if pred == gold: correct += 1
        score = correct / len(samples) * 100 if samples else 0
        ans_rate = answered / len(samples) * 100 if samples else 0
        print(f"GSM8K mini: {correct}/{len(samples)} = {score:.1f}%"
              f" | answered={ans_rate:.0f}%")
        return {"benchmark": "gsm8k_mini", "score": score, "answered": ans_rate}

    def run_hellaswag_mini(model, tokenizer, device, dataset_dir, max_samples=50):
        """HellaSwag mini: commonsense sentence completion, 50 samples."""
        samples = _load_text_dataset(
            dataset_dir, "hellaswag_mini", "Rowan/hellaswag", "validation", max_samples)
        correct = answered = 0
        for s in samples:
            endings = s.get("endings", [])
            opts = "\n".join(f"{chr(65+i)}. {e}" for i, e in enumerate(endings))
            prompt = (f"Context: {s.get('ctx', '')}\n"
                      f"Most appropriate ending?\n{opts}\nAnswer:")
            pred = _extract_mc(_gen_text_answer(model, tokenizer, device, prompt, 10))
            gold = chr(65 + int(s.get("label", 0)))
            if pred:
                answered += 1
                if pred == gold: correct += 1
        score = correct / len(samples) * 100 if samples else 0
        ans_rate = answered / len(samples) * 100 if samples else 0
        print(f"HellaSwag mini: {correct}/{len(samples)} = {score:.1f}%"
              f" | answered={ans_rate:.0f}%")
        return {"benchmark": "hellaswag_mini", "score": score, "answered": ans_rate}

    def run_winogrande_mini(model, tokenizer, device, dataset_dir, max_samples=50):
        """WinoGrande mini: pronoun resolution, 50 samples."""
        samples = _load_text_dataset(
            dataset_dir, "winogrande_mini",
            "allenai/winogrande", "validation", max_samples)
        correct = answered = 0
        for s in samples:
            prompt = (f"Fill in the blank.\nSentence: {s.get('sentence', '')}\n"
                      f"A. {s.get('option1', '')}\nB. {s.get('option2', '')}\nAnswer:")
            pred = _extract_mc(_gen_text_answer(model, tokenizer, device, prompt, 10))
            gold = "A" if s.get("answer", "1") == "1" else "B"
            if pred:
                answered += 1
                if pred == gold: correct += 1
        score = correct / len(samples) * 100 if samples else 0
        ans_rate = answered / len(samples) * 100 if samples else 0
        print(f"WinoGrande mini: {correct}/{len(samples)} = {score:.1f}%"
              f" | answered={ans_rate:.0f}%")
        return {"benchmark": "winogrande_mini", "score": score, "answered": ans_rate}

    def run_truthfulqa_mc(model, tokenizer, device, dataset_dir, max_samples=50):
        """TruthfulQA-MC: factual MCQ, 50 samples."""
        samples = _load_text_dataset(
            dataset_dir, "truthfulqa_mc", "truthful_qa", "validation", max_samples)
        correct = answered = 0
        for s in samples:
            mc1 = s.get("mc1_targets", {})
            choices = mc1.get("choices", [])
            labels  = mc1.get("labels", [])
            if not choices: continue
            cidx = labels.index(1) if 1 in labels else 0
            gold = chr(65 + cidx)
            opts = "\n".join(f"{chr(65+i)}. {c}" for i, c in enumerate(choices))
            prompt = f"Question: {s.get('question', '')}\n{opts}\nAnswer:"
            pred = _extract_mc(_gen_text_answer(model, tokenizer, device, prompt, 10))
            if pred:
                answered += 1
                if pred == gold: correct += 1
        score = correct / len(samples) * 100 if samples else 0
        ans_rate = answered / len(samples) * 100 if samples else 0
        print(f"TruthfulQA-MC: {correct}/{len(samples)} = {score:.1f}%"
              f" | answered={ans_rate:.0f}%")
        return {"benchmark": "truthfulqa_mc", "score": score, "answered": ans_rate}

    log("[Quality] Running text benchmarks...")
    quality_results_text = []
    quality_results_text.append(run_mmlu_pro_mini(model, tokenizer, device, DATASET_DIR))
    quality_results_text.append(run_gsm8k_mini(model, tokenizer, device, DATASET_DIR))
    quality_results_text.append(run_hellaswag_mini(model, tokenizer, device, DATASET_DIR))
    quality_results_text.append(run_winogrande_mini(model, tokenizer, device, DATASET_DIR))
    quality_results_text.append(run_truthfulqa_mc(model, tokenizer, device, DATASET_DIR))
    log("[Quality] Text benchmarks complete")

In [ ]:
import torch
from edge_llm_systems.utils import log

RUN_QUALITY = CONFIG.get("run_quality", False)
input_mode  = CONFIG["input_mode"]

if not RUN_QUALITY:
    print("[Quality] Not enabled, skipping")
elif input_mode != "single_image":
    print("[Quality] Text mode -- vision benchmarks skipped")
else:
    from PIL import Image as PILImage

    def _gen_vision_ans(model, processor, device, image, prompt_text, max_new_tokens):
        """Vision model greedy decoding with image + text input."""
        inputs = processor(images=image, text=prompt_text, return_tensors="pt")
        inputs = {k: v.to(device) if hasattr(v, "to") else v
                  for k, v in inputs.items()}
        with torch.no_grad():
            out = model.generate(
                **inputs, max_new_tokens=max_new_tokens,
                do_sample=False, pad_token_id=processor.tokenizer.eos_token_id,
            )
        new_ids = out[0, inputs["input_ids"].shape[-1]:]
        return processor.tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    def _soft_match(pred, gold):
        """Case-insensitive soft match ignoring punctuation."""
        pred_n = re.sub(r'[^a-z0-9 ]', '', pred.lower())
        gold_n = re.sub(r'[^a-z0-9 ]', '', gold.lower())
        return gold_n in pred_n or pred_n in gold_n

    def _anls(pred, gold, threshold=0.5):
        """ANLS score (Average Normalized Levenshtein Similarity) for DocVQA."""
        import unicodedata
        def norm(s): return unicodedata.normalize("NFD", s.lower().strip())
        p, g = norm(pred), norm(gold)
        if not g: return 0.0
        m_len, n_len = len(p), len(g)
        dp = [[0]*(n_len+1) for _ in range(m_len+1)]
        for i in range(m_len+1): dp[i][0] = i
        for j in range(n_len+1): dp[0][j] = j
        for i in range(1, m_len+1):
            for j in range(1, n_len+1):
                dp[i][j] = (dp[i-1][j-1] if p[i-1]==g[j-1]
                             else 1+min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1]))
        nl = dp[m_len][n_len] / max(m_len, n_len)
        return 1.0 - nl if nl < threshold else 0.0

    def _load_mm_dataset(dataset_dir, benchmark_name, hf_dataset_id,
                          hf_split, max_samples):
        """Load or download multimodal dataset; cache images to disk as PNG."""
        cache_dir = Path(dataset_dir) / "quality_suite" / "multimodal" / benchmark_name
        manifest_path = cache_dir / "manifest.json"
        if manifest_path.exists():
            with open(manifest_path, encoding="utf-8") as f:
                mf = json.load(f)
            if mf.get("sample_count") == max_samples:
                print(f"[Dataset] {benchmark_name}: using cache")
                return mf.get("samples", [])
        print(f"[Dataset] Downloading {benchmark_name}...")
        ds = load_dataset(hf_dataset_id, split=hf_split, trust_remote_code=True)
        raw = [dict(ds[i]) for i in range(min(max_samples, len(ds)))]
        cache_dir.mkdir(parents=True, exist_ok=True)
        samples = []
        for i, s in enumerate(raw):
            entry = {k: v for k, v in s.items()
                     if not isinstance(v, PILImage.Image)}
            if "image" in s and s["image"] is not None:
                img_path = cache_dir / f"{i:04d}.png"
                s["image"].save(str(img_path))
                entry["image_path"] = str(img_path)
            samples.append(entry)
        mf = {"benchmark": benchmark_name, "sample_count": len(samples),
              "sample_ids": [str(i) for i in range(len(samples))],
              "samples": samples}
        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(mf, f, ensure_ascii=False, indent=2)
        print(f"[Dataset] {benchmark_name}: {len(samples)} samples cached")
        return samples

    def _load_img(entry):
        if "image_path" in entry:
            return PILImage.open(entry["image_path"]).convert("RGB")
        return None

    def run_vqav2_mini(model, processor, device, dataset_dir, max_samples=50):
        """VQAv2 mini: general VQA, soft exact match, 50 samples."""
        samples = _load_mm_dataset(
            dataset_dir, "vqav2_mini",
            "HuggingFaceM4/VQAv2", "validation", max_samples)
        correct = answered = 0
        for s in samples:
            img = _load_img(s)
            if img is None: continue
            gold_set = {a.get("answer", "").lower() for a in s.get("answers", [])}
            pred = _gen_vision_ans(model, processor, device, img,
                                   s.get("question", ""), 20).lower()
            answered += 1
            if any(_soft_match(pred, g) for g in gold_set): correct += 1
        score = correct / answered * 100 if answered else 0
        print(f"VQAv2 mini: {correct}/{answered} = {score:.1f}%")
        return {"benchmark": "vqav2_mini", "score": score, "answered": answered}

    def run_mmbench_mini(model, processor, device, dataset_dir, max_samples=50):
        """MMBench mini: multi-dim vision MCQ, 50 samples."""
        samples = _load_mm_dataset(
            dataset_dir, "mmbench_mini",
            "HuggingFaceM4/MMBench", "dev", max_samples)
        correct = answered = 0
        for s in samples:
            img = _load_img(s)
            if img is None: continue
            choices = [s.get("A", ""), s.get("B", ""),
                       s.get("C", ""), s.get("D", "")]
            opts = "\n".join(f"{chr(65+i)}. {c}"
                             for i, c in enumerate(choices) if c)
            prompt = f"{s.get('question', '')}\n{opts}\nAnswer:"
            pred = _extract_mc(_gen_vision_ans(model, processor, device, img, prompt, 10))
            if pred:
                answered += 1
                if pred == s.get("answer", "").upper(): correct += 1
        score = correct / len(samples) * 100 if samples else 0
        print(f"MMBench mini: {correct}/{len(samples)} = {score:.1f}%")
        return {"benchmark": "mmbench_mini", "score": score, "answered": answered}

    def run_mathvista_mini(model, processor, device, dataset_dir, max_samples=30):
        """MathVista mini: visual math MCQ + numeric, 30 samples."""
        samples = _load_mm_dataset(
            dataset_dir, "mathvista_mini",
            "AI4Math/MathVista", "testmini", max_samples)
        correct = answered = 0
        for s in samples:
            img = _load_img(s)
            if img is None: continue
            answer = str(s.get("answer", ""))
            choices = s.get("choices", [])
            answered += 1
            if choices:
                opts = "\n".join(f"{chr(65+i)}. {c}"
                                 for i, c in enumerate(choices))
                pred = _extract_mc(_gen_vision_ans(
                    model, processor, device, img,
                    f"{s.get('question', '')}\n{opts}\nAnswer:", 64))
                if pred and pred == answer.upper(): correct += 1
            else:
                pred_n = _extract_num(_gen_vision_ans(
                    model, processor, device, img,
                    f"{s.get('question', '')}\nAnswer:", 64))
                if pred_n and pred_n == _extract_num(answer): correct += 1
        score = correct / len(samples) * 100 if samples else 0
        print(f"MathVista mini: {correct}/{len(samples)} = {score:.1f}%")
        return {"benchmark": "mathvista_mini", "score": score, "answered": answered}

    def run_textvqa_mini(model, processor, device, dataset_dir, max_samples=30):
        """TextVQA mini: OCR VQA soft match, 30 samples."""
        samples = _load_mm_dataset(
            dataset_dir, "textvqa_mini", "textvqa", "validation", max_samples)
        correct = answered = 0
        for s in samples:
            img = _load_img(s)
            if img is None: continue
            gold_set = {str(a).lower() for a in s.get("answers", [])}
            pred = _gen_vision_ans(
                model, processor, device, img, s.get("question", ""), 20).lower()
            answered += 1
            if any(_soft_match(pred, g) for g in gold_set): correct += 1
        score = correct / answered * 100 if answered else 0
        print(f"TextVQA mini: {correct}/{answered} = {score:.1f}%")
        return {"benchmark": "textvqa_mini", "score": score, "answered": answered}

    def run_docvqa_mini(model, processor, device, dataset_dir, max_samples=30):
        """DocVQA mini: document VQA ANLS score, 30 samples."""
        samples = _load_mm_dataset(
            dataset_dir, "docvqa_mini",
            "nielsr/docvqa_1200_examples", "test", max_samples)
        total_anls = answered = 0
        for s in samples:
            img = _load_img(s)
            if img is None: continue
            answers = s.get("answers", [s.get("answer", "")])
            pred = _gen_vision_ans(
                model, processor, device, img,
                f"{s.get('question', '')}\nAnswer:", 64)
            total_anls += max(_anls(pred, str(a)) for a in answers)
            answered += 1
        score = total_anls / answered * 100 if answered else 0
        print(f"DocVQA mini: ANLS={score:.1f}% | answered={answered}")
        return {"benchmark": "docvqa_mini", "score": score, "answered": answered}

    log("[Quality] Running vision benchmarks...")
    quality_results_vision = []
    quality_results_vision.append(run_vqav2_mini(model, processor, device, DATASET_DIR))
    quality_results_vision.append(run_mmbench_mini(model, processor, device, DATASET_DIR))
    quality_results_vision.append(run_mathvista_mini(model, processor, device, DATASET_DIR))
    quality_results_vision.append(run_textvqa_mini(model, processor, device, DATASET_DIR))
    quality_results_vision.append(run_docvqa_mini(model, processor, device, DATASET_DIR))
    log("[Quality] Vision benchmarks complete")

## Section 8: Results Summary

In [ ]:
import pandas as pd
from pathlib import Path
from edge_llm_systems.utils import log, read_csv_utf8sig
from edge_llm_systems.models import MODEL_REGISTRY

log("[Results] Experiment output files:")

# model_slug 在 Section 6 中定义；如果 Section 6 被跳过则从 model_key 重新推导
if "model_slug" not in dir():
    model_slug = MODEL_REGISTRY.get(model_key, model_key).split("/")[-1]

model_result_dir = Path(f"{RESULTS_DIR}/{model_slug}")

# 列出本次实验产出的所有文件
for subdir in ["raw_runs", "group_summary", "exp_info/environment",
               "exp_info/model", "exp_info/run_config"]:
    subpath = model_result_dir / subdir
    if subpath.exists():
        for f in sorted(subpath.glob("*")):
            if f.is_file() and f.name != ".gitkeep":
                print(f"  {f.relative_to(model_result_dir)}")

# 读取最新 group_summary CSV 并打印均值性能表
# No matplotlib -- pandas DataFrame.to_string() only
summary_files = sorted((model_result_dir / "group_summary").glob("*.csv"))

if summary_files:
    latest_summary = summary_files[-1]
    rows = read_csv_utf8sig(str(latest_summary))
    if rows:
        df = pd.DataFrame(rows)
        # 只展示均值行（run_index == '0'）
        df_mean = df[df["run_index"].astype(str) == "0"].copy()
        display_cols = [
            "model_id", "input_mode",
            "prompt_len", "gen_len", "image_resolution",
            "ttft_ms", "tpot_ms", "tokens_s",
            "peak_mem_mb", "kv_pkv_final_mb", "status",
        ]
        display_cols = [c for c in display_cols if c in df_mean.columns]
        print(f"\n[Performance Summary] Mean results — {latest_summary.name}:")
        print(df_mean[display_cols].to_string(index=False))
    else:
        print("[Results] Summary CSV is empty")
else:
    print("[Results] No summary CSV found in", model_result_dir / "group_summary")

log("[Results] Experiment complete")